In [ ]:
import pandas as pd
import yaml
import io
import openpyxl
import csv

In [ ]:
excel_file = 'DV.xlsx'

xl_file = pd.ExcelFile(excel_file)
sheet_names = xl_file.sheet_names
print(f"Các sheet trong file: {sheet_names}")

sheet_list = ['Production, Raw Mat',
'Energy, GHG',
'Water',
'Waste',
'Emission']

for sheet in sheet_names:
    if sheet in sheet_list:
        df = pd.read_excel(excel_file, sheet_name=sheet)
        print(f"Sheet: {sheet}")
        print(f"Kích thước: {df.shape[0]} dòng, {df.shape[1]} cột")
        print(df)
    print("\n")

Các sheet trong file: ['Business', 'Scope', 'Production, Raw Mat', 'Energy, GHG', 'Energy, GHG (2)', 'Water', 'Waste', 'Emission', 'CO2 SAY PHUN']




Sheet: Production, Raw Mat
Kích thước: 20 dòng, 19 cột
                                           Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                    Production (DV Tile + YB Powder)       unit        Jan   
1                  Production (DV Tile) (Sản phẩm ĐV)        ton   9609.931   
2       Production (YB Powder) (bột bán cho Yên Bình)        ton          0   
3                          Total Production (DV + YB)        ton   9609.931   
4                                                 NaN        NaN        NaN   
5                                        Raw Material       unit        Jan   
6                          Limestone/ Đá vôi (canxit)        ton    1185.21   
7                                       Clay/ Đất sét        ton    6280.95   
8                                           Sand/ Cát        ton        NaN   
9   

Energy, GHG

In [ ]:
# sheet 'Energy, GHG'
df_energy = pd.read_excel(excel_file, sheet_name='Energy, GHG')
print(df_energy.head(10))

                                     Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                            Energy Consumption       unit        Jan   
1                            Coal/ Than 5A (DV)        Ton   1233.522   
2                          Coal/ Than 4A.1 (DV)        Ton        443   
3  Coal/ Than import coal (DV) (than nhập khẩu)        Ton          0   
4   Than cám qua sàng / Than 5A under screen DV        Ton        114   
5                            Coal/ Than 5A (YB)        Ton          0   
6                          Coal/ Than 4A.1 (YB)        Ton          0   
7  Coal/ Than import coal (YB) (than nhập khẩu)        Ton          0   
8   Than cám qua sàng / Than 5A under screen YB        Ton          0   
9                         Coal/ Than 5A (DV+YB)        Ton   1233.522   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        Feb        Mar        Apr        May        Jun        Jul   
1        983   2414.661   2198.795   2197.301    2170.

In [ ]:
def find_sections(df, section_names):
    section_markers = {}
    for idx, row in df.iterrows():
        row_str = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ''
        for name in section_names:
            if row_str == name:
                section_markers[name] = idx
    return section_markers

In [ ]:
energy_df = df_energy.reset_index(drop=True)

section_names = ['Energy Consumption', 'Greenhouse Gas Emission', 'Conversion Factor', 'Energy Cost']
section_markers = find_sections(energy_df, section_names)

sorted_sections = sorted([(idx, s) for s, idx in section_markers.items() if idx is not None])

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(energy_df)
    df_part = energy_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())



Energy Consumption
65 hàng x 22 cột
0                            Energy Consumption unit       Jan     Feb       Mar       Apr       May      Jun       Jul       Aug       Sep       Oct      Nov         Dec  YTD        Q1        Q2        Q3          Q4  NaN  NaN  NaN
0                            Coal/ Than 5A (DV)  Ton  1233.522     983  2414.661  2198.795  2197.301  2170.99  2442.916  2148.882  1718.913  1753.955   1649.3    2061.215  NaN  4631.183  6567.086  6310.711     5464.47  NaN  NaN  NaN
1                          Coal/ Than 4A.1 (DV)  Ton       443  251.25   543.069    88.335    774.19  319.418    508.88   630.609   444.559      18.9   113.76     369.092  NaN  1237.319  1181.943  1584.048     501.752  NaN  NaN  NaN
2  Coal/ Than import coal (DV) (than nhập khẩu)  Ton         0   92.48   430.117   722.734         0   434.63    375.27     77.21         0    550.88  352.255     533.023  NaN   522.597  1157.364    452.48    1436.158  NaN  NaN  NaN
3   Than cám qua sàng / Than 5A 

In [ ]:
import re

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(energy_df)
    df_part = energy_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

    safe_section = re.sub(r'[^A-Za-z0-9_]+', '_', section)
    df_part_clean.to_csv(f"{safe_section}.csv", index=False)

Energy Consumption
65 hàng x 22 cột
0                            Energy Consumption unit       Jan     Feb       Mar       Apr       May      Jun       Jul       Aug       Sep       Oct      Nov         Dec  YTD        Q1        Q2        Q3          Q4  NaN  NaN  NaN
0                            Coal/ Than 5A (DV)  Ton  1233.522     983  2414.661  2198.795  2197.301  2170.99  2442.916  2148.882  1718.913  1753.955   1649.3    2061.215  NaN  4631.183  6567.086  6310.711     5464.47  NaN  NaN  NaN
1                          Coal/ Than 4A.1 (DV)  Ton       443  251.25   543.069    88.335    774.19  319.418    508.88   630.609   444.559      18.9   113.76     369.092  NaN  1237.319  1181.943  1584.048     501.752  NaN  NaN  NaN
2  Coal/ Than import coal (DV) (than nhập khẩu)  Ton         0   92.48   430.117   722.734         0   434.63    375.27     77.21         0    550.88  352.255     533.023  NaN   522.597  1157.364    452.48    1436.158  NaN  NaN  NaN
3   Than cám qua sàng / Than 5A 

Production, Raw Mat

In [ ]:
# sheet 'Energy, GHG'
df_production = pd.read_excel(excel_file, sheet_name='Production, Raw Mat')
print(df_production.head(10))

                                      Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0               Production (DV Tile + YB Powder)       unit        Jan   
1             Production (DV Tile) (Sản phẩm ĐV)        ton   9609.931   
2  Production (YB Powder) (bột bán cho Yên Bình)        ton          0   
3                     Total Production (DV + YB)        ton   9609.931   
4                                            NaN        NaN        NaN   
5                                   Raw Material       unit        Jan   
6                     Limestone/ Đá vôi (canxit)        ton    1185.21   
7                                  Clay/ Đất sét        ton    6280.95   
8                                      Sand/ Cát        ton        NaN   
9                              Gypsum/ Thạch cao        ton        NaN   

    Unnamed: 3   Unnamed: 4   Unnamed: 5   Unnamed: 6   Unnamed: 7  \
0          Feb          Mar          Apr          May          Jun   
1  6310.080933  17732.97347  15795.45537  160

In [ ]:
pro_df = df_production.reset_index(drop=True)
section_names = ['Production (DV Tile + YB Powder)', 'Raw Material']
section_markers = find_sections(pro_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(pro_df)
    df_part = pro_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

Production (DV Tile + YB Powder)
4 hàng x 19 cột
0               Production (DV Tile + YB Powder) unit       Jan          Feb          Mar          Apr          May          Jun         Jul          Aug          Sep          Oct          Nov          Dec          YTD          Q1           Q2           Q3           Q4
0             Production (DV Tile) (Sản phẩm ĐV)  ton  9609.931  6310.080933  17732.97347  15795.45537  16091.79977  16408.01583  18454.3238  16179.63025  12136.76037  12464.84122  12324.14555  15580.26882  169088.2264  33652.9854  48295.27097  46770.71442  40369.25559
1  Production (YB Powder) (bột bán cho Yên Bình)  ton         0            0            0            0            0          NaN         NaN          NaN          NaN          NaN          NaN     423.6674     423.6674           0            0            0     423.6674
2                     Total Production (DV + YB)  ton  9609.931  6310.080933  17732.97347  15795.45537  16091.79977  16408.01583  18454.3238 

Water

In [ ]:
water_df = pd.read_excel(excel_file, sheet_name='Water')
print(water_df.head(10))

                                     Unnamed: 0 Unnamed: 1 Unnamed: 2  \
0                              Water Withdrawal       unit        Jan   
1          Surface water (Lake) (DV) (Nước mặt)         m3       4987   
2          Surface water (Lake) (YB) (Nước Mặt)         m3          0   
3  Surface water (Lake) (DV+YB) (tổng nước mặt)         m3       4987   
4                     Ground water (wells) (DV)         m3          0   
5                     Tap wate (DV) (Nước sạch)         m3        658   
6                   Total Water Withdrawal (DV)         m3       5645   
7                   Total Water Withdrawal (YB)         m3          0   
8                Total Water Withdrawal (DV+YB)         m3       5645   
9                Specific Water Withdrawal (DV)     m3/ton   0.587413   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        Feb        Mar        Apr        May        Jun        Jul   
1       3058       6675       6392       6393       62

In [ ]:
water_df = water_df.reset_index(drop=True)
section_names = ['Water Withdrawal', 'Water Recycle']
section_markers = find_sections(water_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])
for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(water_df)
    df_part = water_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(10).to_string())

Water Withdrawal
12 hàng x 19 cột
0                              Water Withdrawal    unit       Jan       Feb       Mar       Apr       May       Jun       Jul       Aug       Sep       Oct       Nov       Dec      YTD       Q1       Q2       Q3       Q4
0          Surface water (Lake) (DV) (Nước mặt)      m3      4987      3058      6675      6392      6393      6254      6934      6241      5438      6081      6741      7894    73088    14720    19039    18613    20716
1          Surface water (Lake) (YB) (Nước Mặt)      m3         0         0         0         0         0         0         0         0         0         0         0         0        0        0        0        0        0
2  Surface water (Lake) (DV+YB) (tổng nước mặt)      m3      4987      3058      6675      6392      6393      6254      6934      6241      5438      6081      6741      7894    73088    14720    19039    18613    20716
3                     Ground water (wells) (DV)      m3         0         0       

#Waste

In [ ]:
waste_df = pd.read_excel(excel_file, sheet_name='Waste')
print(waste_df.head(10))

                                          Unnamed: 0               Unnamed: 1  \
0          Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI                      Jan   
1                                                NaN  Incoming Stock/ Tồn đầu   
2                                                NaN                      NaN   
3  Contaminated Cloth/ Bao gồm giẻ lau, gang tay ...                   0.3472   
4  Used oil/ Tất cả các loại dầu đã qua sử dụng thải                   5.8526   
5  Package contaminated / Vỏ thùng sơn, vỏ thùng ...                    2.413   
6  Electronic equipment / Các thiết bị điện thải ...                   0.1484   
7  Insulation/ Vật liệu cách nhiệt (amiăng thải, ...                   1.6998   
8                                  Đất dính hóa chất                   0.2316   
9                     Cặn rắn từ quá trình sử lý men                     0.04   

                             Unnamed: 2                  Unnamed: 3  \
0                                   N

In [ ]:
df_waste = waste_df.reset_index(drop=True)
df_waste.info()
df_waste.head(10)
section_names = ['Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI', 'Non Hazardous Waste (ton)/ CHẤT THẢI THÔNG THƯỜNG']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 67 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   24 non-null     object 
 1   Unnamed: 1   24 non-null     object 
 2   Unnamed: 2   24 non-null     object 
 3   Unnamed: 3   10 non-null     object 
 4   Unnamed: 4   4 non-null      object 
 5   Unnamed: 5   6 non-null      object 
 6   Unnamed: 6   24 non-null     object 
 7   Unnamed: 7   24 non-null     object 
 8   Unnamed: 8   8 non-null      object 
 9   Unnamed: 9   6 non-null      object 
 10  Unnamed: 10  6 non-null      object 
 11  Unnamed: 11  24 non-null     object 
 12  Unnamed: 12  23 non-null     object 
 13  Unnamed: 13  12 non-null     object 
 14  Unnamed: 14  4 non-null      object 
 15  Unnamed: 15  6 non-null      object 
 16  Unnamed: 16  24 non-null     object 
 17  Unnamed: 17  23 non-null     object 
 18  Unnamed: 18  6 non-null      object 
 19  Unnamed: 1

In [ ]:
section_names = [
    'Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI',
    'Non Hazardous Waste (ton)/ CHẤT THẢI THÔNG THƯỜNG'
]
section_markers = find_sections(waste_df, section_names)
sorted_sections = sorted([(section_markers[s], s) for s in section_names if s in section_markers])

all_flat = []

for i, (start_idx, section) in enumerate(sorted_sections):
    end_idx = sorted_sections[i + 1][0] if i < len(sorted_sections) - 1 else len(waste_df)
    df_part = waste_df.iloc[start_idx:end_idx].copy()
    df_part_clean = pd.DataFrame(df_part.iloc[1:].values, columns=df_part.iloc[0])
    print(section)
    print(len(df_part_clean), "hàng x", len(df_part_clean.columns), "cột")
    print(df_part_clean.head(3).to_string())

    data = df_part_clean.iloc[3:].reset_index(drop=True)
    data.columns = [f"col_{i}" for i in range(data.shape[1])]

    for idx, row in data.iterrows():
        waste_category_en = row['col_0']
        for m, month in enumerate(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
            base = 1 + m * 5
            record = {
                'section': section,
                'waste_category_en': waste_category_en,
                'report_month': month,
                'incoming_stock': row.get(f'col_{base}'),
                'waste_generated': row.get(f'col_{base+1}'),
                'reuse_recycle': row.get(f'col_{base+2}'),
                'incineration': row.get(f'col_{base+3}'),
                'landfill': row.get(f'col_{base+4}'),
            }
            all_flat.append(record)

df_flat = pd.DataFrame(all_flat)
print(df_flat.head(10))

Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI
14 hàng x 67 cột
0                              Hazardous Waste (ton)/ CHẤT THẢI NGUY HẠI                      Jan                                   NaN                         NaN                          NaN                           NaN                      Feb                                   NaN                         NaN                          NaN                           NaN                      Mar                                   NaN                         NaN                          NaN                           NaN                      Apr                                   NaN                         NaN                          NaN                           NaN                      May                                   NaN                         NaN                          NaN                           NaN                      Jun                                   NaN                         NaN                          Na

Emission

| STT | Tên cột (Field Name)         | Kiểu dữ liệu   | Ý nghĩa & Ví dụ từ ảnh                                   | Ghi chú                                    |
|-----|------------------------------|---------------|---------------------------------------------------------|--------------------------------------------|
| 1   | stack_no                     | String        | Tên ống khói (Sấy phun 1, Sấy phun 2...)                | Khóa chính để phân loại nguồn thải          |
| 2   | process                      | String        | Quy trình sản xuất liên quan                             |                                            |
| 3   | diameter                     | Float         | Đường kính ống khói (m)                                 | Thông số kỹ thuật cố định                   |
| 4   | sampling_month_year          | String/Date   | Tháng/Năm lấy mẫu (Ví dụ: 2/12/2019)                    | Cột quan trọng để phân biệt các đợt đo      |
| 5   | stack_temperature            | Float         | Nhiệt độ ống khói (°C)                                  |                                            |
| 6   | oxygen_pct                   | Float         | % Oxygen trong khí thải                                  |                                            |
| 7   | gas_velocity                 | Float         | Vận tốc khí (m/s)                                       |                                            |
| 8   | flow_rate                    | Float         | Lưu lượng khí (m3/min)                                  |                                            |
| 9   | dust_emission_actual_o2      | Float         | Kết quả đo bụi thực tế (mg/m3)                          |                                            |
| 10  | dust_emission_rate           | Float         | Tỉ lệ bụi phát thải (kg/hr)                             |                                            |
| 11  | operation_hour               | Float         | Thời gian hoạt động (hr)                                |                                            |
| 12  | dust_emission_total          | Float         | Tổng lượng bụi phát thải (kg)                           | Kết quả của (Cột 10 * Cột 11)              |

In [ ]:
df_emission = pd.read_excel(excel_file, sheet_name='Emission')
print(f"Kích thước dữ liệu gốc: {df_emission.shape}")
display(df_emission.head())

Kích thước dữ liệu gốc: (54, 116)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Dust/ Bụi,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter/ Đường kính ống khói,Sampling \n(Month/Year),Stack \nTemperature,%Oxygen,Gas Velocity,Flow rate/ Lưu lượng,Dust Emission\nActual O2/ Kết quả đo bụi,Dust Emission \nrate/ Tỉ lệ bụi,...,Flow rate,Dust Emission\nActual O2,Dust Emission \nrate,Operation \nHour,Dust Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Sấy phun 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,41,3.485,...,1416.666667,43.4,3.689,660,2434.74,NaN,NaN,NaN,NaN,NaN
4,Sấy phun 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,43,3.655,...,1416.666667,42.3,3.5955,436,1567.638,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Tìm index của các dòng chứa chữ 'Stack No.'
stack_no_rows = df_emission[df_emission['Unnamed: 0'].astype(str).str.strip().str.lower() == 'stack no.'].index.tolist()
print("Vị trí các dòng 'Stack No.' tìm thấy:", stack_no_rows)

#Tiến hành cắt bảng
tables = {}
for i in range(len(stack_no_rows)):
    # Dòng bắt đầu của bảng là dòng NGAY TRÊN dòng 'Stack No.'
    start_idx = stack_no_rows[i] - 1
    # Lấy tên bảng
    table_name = str(df_emission.iloc[start_idx, 0]).strip()
    end_idx = stack_no_rows[i+1] - 1 if i + 1 < len(stack_no_rows) else len(df_emission)
    df_sub = df_emission.iloc[start_idx:end_idx].reset_index(drop=True)

    # Dọn dẹp các dòng trống rác cuối bảng
    df_sub = df_sub.dropna(how='all')

    tables[table_name] = df_sub

print(f"\n Đã tách thành công {len(tables)} bảng:")
for name, df in tables.items():
    print(f" - Bảng '{name}': {df.shape[0]} dòng, {df.shape[1]} cột")

    # In nội dung của bảng
    display(df)
    print("\n")

Vị trí các dòng 'Stack No.' tìm thấy: [1, 19, 37]

 Đã tách thành công 3 bảng:
 - Bảng 'Dust/ Bụi': 17 dòng, 116 cột


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Dust/ Bụi,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter/ Đường kính ống khói,Sampling \n(Month/Year),Stack \nTemperature,%Oxygen,Gas Velocity,Flow rate/ Lưu lượng,Dust Emission\nActual O2/ Kết quả đo bụi,Dust Emission \nrate/ Tỉ lệ bụi,...,Flow rate,Dust Emission\nActual O2,Dust Emission \nrate,Operation \nHour,Dust Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Sấy phun 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,41,3.485,...,1416.666667,43.4,3.689,660,2434.74,NaN,NaN,NaN,NaN,NaN
4,Sấy phun 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,43,3.655,...,1416.666667,42.3,3.5955,436,1567.638,NaN,NaN,NaN,NaN,NaN
5,"Ống khói lò nung xương 1,2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.333333,52,0.5408,...,154,45.1,0.416724,744,310.042656,NaN,NaN,NaN,NaN,NaN
6,"Ống khói lò nung xương 3,4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.333333,53,0.5989,...,158.333333,42.6,0.4047,744,301.0968,NaN,NaN,NaN,NaN,NaN
7,"Ống khói lò nung xương 5,6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,50,0.5265,...,194.166667,43.9,0.511435,744,380.50764,NaN,NaN,NaN,NaN,NaN
8,Ống khói lò nung men số 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.666667,51,0.51612,...,178.333333,43.7,0.46759,744,347.88696,NaN,NaN,NaN,NaN,NaN
9,Ống khói lò nung men số 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,52,0.44148,...,155.666667,44.2,0.412828,744,307.144032,NaN,NaN,NaN,NaN,NaN




 - Bảng 'Sox/ Khí Sox': 17 dòng, 116 cột


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Sox/ Khí Sox,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling \n(Month/Year),Stack \nTemperature,%Oxygen,Gas Velocity,Flow rate,SOx Emission\nActual O2,SOx Emission \nrate,...,Flow rate,SOx Emission\nActual O2,SOx Emission \nrate,Operation \nHour,SOx Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Sấy phun 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,15.72,1.3362,...,1416.666667,94.2,8.007,660,5284.62,NaN,NaN,NaN,NaN,NaN
4,Sấy phun 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,18.34,1.5589,...,1416.666667,94.7,8.0495,436,3509.582,NaN,NaN,NaN,NaN,NaN
5,"Ống khói lò nung xương 1,2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.333333,83.84,0.871936,...,154,92.4,0.853776,744,635.209344,NaN,NaN,NaN,NaN,NaN
6,"Ống khói lò nung xương 3,4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.333333,84.71,0.957223,...,158.333333,96.1,0.91295,744,679.2348,NaN,NaN,NaN,NaN,NaN
7,"Ống khói lò nung xương 5,6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,80.19,0.844401,...,194.166667,93.8,1.09277,744,813.02088,NaN,NaN,NaN,NaN,NaN
8,Ống khói lò nung men số 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.666667,6.38,0.064566,...,178.333333,96.2,1.02934,744,765.82896,NaN,NaN,NaN,NaN,NaN
9,Ống khói lò nung men số 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,6.42,0.054506,...,155.666667,97.8,0.913452,744,679.608288,NaN,NaN,NaN,NaN,NaN




 - Bảng 'Nox': 18 dòng, 116 cột


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
0,Nox,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling \n(Month/Year),Stack \nTemperature,%Oxygen,Gas Velocity,Flow rate,NOx Emission\nActual O2,NOx Emission \nrate,...,Flow rate,NOx Emission\nActual O2,NOx Emission \nrate,Operation \nHour,NOx Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Sấy phun 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,4.4,0.374,...,1416.666667,71.6,6.086,660,4016.76,NaN,NaN,NaN,NaN,NaN
4,Sấy phun 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,4.6,0.391,...,1416.666667,67.9,5.7715,436,2516.374,NaN,NaN,NaN,NaN,NaN
5,"Ống khói lò nung xương 1,2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.333333,5.1,0.05304,...,154,72.7,0.671748,744,499.780512,NaN,NaN,NaN,NaN,NaN
6,"Ống khói lò nung xương 3,4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.333333,4.6,0.05198,...,158.333333,68.9,0.65455,744,486.9852,NaN,NaN,NaN,NaN,NaN
7,"Ống khói lò nung xương 5,6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,4.8,0.050544,...,194.166667,70.4,0.82016,744,610.19904,NaN,NaN,NaN,NaN,NaN
8,Ống khói lò nung men số 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.666667,5.7,0.057684,...,178.333333,72.4,0.77468,744,576.36192,NaN,NaN,NaN,NaN,NaN
9,Ống khói lò nung men số 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,5.5,0.046695,...,155.666667,70.7,0.660338,744,491.291472,NaN,NaN,NaN,NaN,NaN


In [ ]:
import os

# Danh sách tên file
output_filenames = ['Emission_Dust.csv', 'Emission_Nox.csv', 'Emission_Sox.csv']
for df, filename in zip(tables.values(), output_filenames):
    # Xuất ra CSV
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"Đã xuất thành công file: {filename}")

Đã xuất thành công file: Emission_Dust.csv
Đã xuất thành công file: Emission_Nox.csv
Đã xuất thành công file: Emission_Sox.csv
